# GenAI Tokenomics — Live Inference Demo

Use this notebook with an OpenAI-compatible LLM or vLLM endpoint, directly or through APISIX. It demonstrates how input tokens, output tokens, latency, throughput, cost, and agent loops affect response generation.

## Demo storyline (10–15 minutes)

1. Call the model and inspect `prompt_tokens`, `completion_tokens`, and `total_tokens`.
2. Stream a response and measure time to first token (TTFT) and generation speed.
3. Change `max_tokens` to show how output length affects latency and consumption.
4. Add context to show how input tokens increase prefill work.
5. Convert token usage into cost or internal capacity estimates.
6. Show why an agent loop multiplies token consumption.

> Keep secrets out of the notebook. The API token is requested with `getpass()` and is not saved.

In [ ]:
# If needed, uncomment this once:
# %pip install requests pandas matplotlib

import json
import os
import time
import uuid
from getpass import getpass

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## 1. Configure the endpoint

Examples:

- Direct vLLM: `https://llm.example.com/v1`
- Through APISIX: `https://gateway.example.com/llm/v1`

Enter the base URL ending at `/v1`, not the full `/chat/completions` URL. For an internal CA, set `CA_BUNDLE` to its PEM path. Avoid disabling TLS verification in a real demo.

In [ ]:
BASE_URL = os.getenv("LLM_BASE_URL", "https://your-llm-or-apisix-host/v1").rstrip("/")
MODEL = os.getenv("LLM_MODEL", "your-model-name")
CA_BUNDLE = os.getenv("CA_BUNDLE")
VERIFY_TLS = CA_BUNDLE if CA_BUNDLE else True
REQUEST_TIMEOUT_SECONDS = 180

API_KEY = os.getenv("LLM_API_KEY") or getpass("Inference API token: " )
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

MODELS_URL = f"{BASE_URL}/models"
CHAT_URL = f"{BASE_URL}/chat/completions"

print("Base URL:", BASE_URL)
print("Model:", MODEL)
print("TLS verification:", VERIFY_TLS)

In [ ]:
# Optional connectivity check and model discovery
models_response = requests.get(
    MODELS_URL, headers=HEADERS, timeout=30, verify=VERIFY_TLS
)
models_response.raise_for_status()
available_models = [item.get("id") for item in models_response.json().get("data", [])]
print("Available models:", available_models)

# If your configured model is a placeholder, select one explicitly:
# MODEL = available_models[0]

## 2. Helper functions

The endpoint's own `usage` object is the source of truth. Local word counts are not token counts. The helpers also attach an `X-Demo-Request-ID`, making it easier to correlate a notebook request with APISIX, application, Loki, or tracing data.

In [ ]:
experiment_rows = []

def _usage_values(usage):
    usage = usage or {}
    prompt = usage.get("prompt_tokens")
    completion = usage.get("completion_tokens")
    total = usage.get("total_tokens")
    if total is None and prompt is not None and completion is not None:
        total = prompt + completion
    return prompt, completion, total

def call_chat(label, messages, max_tokens=200, temperature=0.0, extra_body=None):
    demo_request_id = str(uuid.uuid4())
    headers = {**HEADERS, "X-Demo-Request-ID": demo_request_id}
    payload = {
        "model": MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        **(extra_body or {}),
    }

    started = time.perf_counter()
    response = requests.post(
        CHAT_URL, headers=headers, json=payload,
        timeout=REQUEST_TIMEOUT_SECONDS, verify=VERIFY_TLS
    )
    elapsed = time.perf_counter() - started
    response.raise_for_status()
    body = response.json()

    usage = body.get("usage") or {}
    prompt_tokens, completion_tokens, total_tokens = _usage_values(usage)
    content = body.get("choices", [{}])[0].get("message", {}).get("content", "")
    finish_reason = body.get("choices", [{}])[0].get("finish_reason")

    row = {
        "experiment": label,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "ttft_s": None,
        "generation_s": None,
        "total_latency_s": elapsed,
        "tokens_per_second": (completion_tokens / elapsed) if completion_tokens else None,
        "max_tokens": max_tokens,
        "finish_reason": finish_reason,
        "demo_request_id": demo_request_id,
        "gateway_request_id": response.headers.get("x-request-id"),
        "traceparent": response.headers.get("traceparent"),
    }
    experiment_rows.append(row)
    return row, content, body

def call_chat_stream(label, messages, max_tokens=200, temperature=0.0, extra_body=None):
    demo_request_id = str(uuid.uuid4())
    headers = {**HEADERS, "X-Demo-Request-ID": demo_request_id}
    payload = {
        "model": MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "stream": True,
        "stream_options": {"include_usage": True},
        **(extra_body or {}),
    }

    started = time.perf_counter()
    first_token_at = None
    chunks = []
    usage = {}
    finish_reason = None

    with requests.post(
        CHAT_URL, headers=headers, json=payload, stream=True,
        timeout=REQUEST_TIMEOUT_SECONDS, verify=VERIFY_TLS
    ) as response:
        response.raise_for_status()
        gateway_request_id = response.headers.get("x-request-id")
        traceparent = response.headers.get("traceparent")

        for raw_line in response.iter_lines(decode_unicode=True):
            if not raw_line or not raw_line.startswith("data:"):
                continue
            data = raw_line[5:].strip()
            if data == "[DONE]":
                break
            event = json.loads(data)
            if event.get("usage"):
                usage = event["usage"]
            for choice in event.get("choices", []):
                piece = choice.get("delta", {}).get("content") or ""
                if piece:
                    if first_token_at is None:
                        first_token_at = time.perf_counter()
                    chunks.append(piece)
                    print(piece, end="", flush=True)
                if choice.get("finish_reason"):
                    finish_reason = choice["finish_reason"]

    ended = time.perf_counter()
    print()
    ttft = (first_token_at - started) if first_token_at else None
    generation_s = (ended - first_token_at) if first_token_at else None
    prompt_tokens, completion_tokens, total_tokens = _usage_values(usage)

    row = {
        "experiment": label,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "ttft_s": ttft,
        "generation_s": generation_s,
        "total_latency_s": ended - started,
        "tokens_per_second": (completion_tokens / generation_s) if completion_tokens and generation_s else None,
        "max_tokens": max_tokens,
        "finish_reason": finish_reason,
        "demo_request_id": demo_request_id,
        "gateway_request_id": gateway_request_id,
        "traceparent": traceparent,
    }
    experiment_rows.append(row)
    return row, "".join(chunks), usage

def results_table():
    return pd.DataFrame(experiment_rows)

## 3. Demo A — One request, three token measurements

Explain that input tokens are processed during **prefill**, while output tokens are produced sequentially during **generation**. For self-hosted vLLM, tokens still consume GPU time and capacity even when there is no per-token vendor bill.

In [ ]:
baseline_messages = [
    {"role": "system", "content": "You are a concise data-platform assistant."},
    {"role": "user", "content": "Explain APISIX observability in exactly four bullet points."},
]

baseline_row, baseline_answer, baseline_raw = call_chat(
    "baseline_non_streaming", baseline_messages, max_tokens=180
)
display(Markdown(baseline_answer))
display(pd.DataFrame([baseline_row]))
print("Raw usage object:", baseline_raw.get("usage"))

### What to say

- `prompt_tokens`: system prompt + user prompt + any history/context.
- `completion_tokens`: tokens generated in the answer.
- `total_tokens`: normally prompt + completion.
- `max_tokens` is a ceiling, not a promise that the model will use all of them.
- `completion_tokens_details` or `prompt_tokens_details` may be absent or null; the three top-level usage counters remain sufficient for basic monitoring.

## 4. Demo B — Streaming, TTFT, and tokens per second

TTFT is what a user feels before the answer begins. Generation speed describes how quickly the remaining output arrives.

In [ ]:
stream_row, stream_answer, stream_usage = call_chat_stream(
    "baseline_streaming", baseline_messages, max_tokens=180
)
display(pd.DataFrame([stream_row]))
print("Streaming usage object:", stream_usage)

> If streaming usage is empty, check whether your vLLM version/provider supports `stream_options={"include_usage": true}` and whether APISIX passes the final SSE usage event unchanged.

## 5. Demo C — Output budget experiment

The same task is called with different output ceilings. A low ceiling can reduce consumption but may stop the response with `finish_reason=length`. This cell makes three real calls.

In [ ]:
output_budget_messages = [
    {"role": "system", "content": "You explain technical topics clearly and practically."},
    {"role": "user", "content": "Explain five methods to reduce token consumption in a production RAG system, with examples."},
]

for budget in [50, 150, 400]:
    row, _, _ = call_chat(
        f"output_budget_{budget}", output_budget_messages, max_tokens=budget
    )
    print(f"Completed max_tokens={budget}; finish_reason={row['finish_reason']}")

budget_df = results_table().query("experiment.str.startswith('output_budget_')", engine="python")
display(budget_df[[
    "experiment", "prompt_tokens", "completion_tokens",
    "total_latency_s", "max_tokens", "finish_reason"
]])

In [ ]:
budget_plot = budget_df.sort_values("max_tokens")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(budget_plot["max_tokens"].astype(str), budget_plot["completion_tokens"])
axes[0].set(title="Output budget vs generated tokens", xlabel="max_tokens", ylabel="completion tokens")
axes[1].bar(budget_plot["max_tokens"].astype(str), budget_plot["total_latency_s"], color="#ff8c42")
axes[1].set(title="Output budget vs latency", xlabel="max_tokens", ylabel="seconds")
plt.tight_layout()
plt.show()

## 6. Demo D — Input context experiment

This shows why sending entire documents or full chat history is expensive. The question stays constant while the supplied context grows. This cell makes three real calls.

In [ ]:
context_unit = (
    "APISIX receives the request, applies authentication and routing, forwards it to vLLM, "
    "and exports logs, metrics, and trace context for observability. "
)

for label, repeats in [("small", 1), ("medium", 20), ("large", 80)]:
    supplied_context = context_unit * repeats
    messages = [
        {"role": "system", "content": "Answer only from the supplied context."},
        {"role": "user", "content": f"Context:\n{supplied_context}\n\nQuestion: Which component forwards the request to vLLM?"},
    ]
    call_chat(f"input_context_{label}", messages, max_tokens=40)

context_df = results_table().query("experiment.str.startswith('input_context_')", engine="python")
display(context_df[[
    "experiment", "prompt_tokens", "completion_tokens", "total_latency_s"
]])

In [ ]:
context_plot = context_df.sort_values("prompt_tokens")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(context_plot["prompt_tokens"], context_plot["total_latency_s"], marker="o", linewidth=2)
for _, item in context_plot.iterrows():
    ax.annotate(item["experiment"].replace("input_context_", ""),
                (item["prompt_tokens"], item["total_latency_s"]), xytext=(5, 5), textcoords="offset points")
ax.set(title="Input context and end-to-end latency", xlabel="prompt tokens", ylabel="seconds")
ax.grid(alpha=0.25)
plt.show()

## 7. Convert tokens into cost or capacity

For a hosted API, enter its input and output prices per one million tokens. For self-hosted vLLM, leave prices at zero and use total tokens plus latency as capacity indicators—or replace the rates with your internal chargeback values.

In [ ]:
INPUT_PRICE_PER_MILLION = 0.0   # Replace if applicable
OUTPUT_PRICE_PER_MILLION = 0.0  # Replace if applicable
REQUESTS_PER_DAY = 10_000

def add_cost_columns(frame):
    result = frame.copy()
    result["input_cost"] = result["prompt_tokens"].fillna(0) / 1_000_000 * INPUT_PRICE_PER_MILLION
    result["output_cost"] = result["completion_tokens"].fillna(0) / 1_000_000 * OUTPUT_PRICE_PER_MILLION
    result["request_cost"] = result["input_cost"] + result["output_cost"]
    return result

cost_df = add_cost_columns(results_table())
display(cost_df[[
    "experiment", "prompt_tokens", "completion_tokens", "request_cost"
]])

if not cost_df.empty:
    average_tokens = cost_df["total_tokens"].dropna().mean()
    average_cost = cost_df["request_cost"].mean()
    print(f"Estimated tokens/day: {average_tokens * REQUESTS_PER_DAY:,.0f}")
    print(f"Estimated cost/day:  {average_cost * REQUESTS_PER_DAY:,.2f}")
    print(f"Estimated cost/30d:  {average_cost * REQUESTS_PER_DAY * 30:,.2f}")

## 8. Agent-loop multiplier

An agent may call the model for planning, tool selection, observation, reflection, and the final answer. The table below estimates how repeated calls multiply tokens and cost without issuing more requests.

In [ ]:
observed = results_table().dropna(subset=["prompt_tokens", "completion_tokens"]).iloc[0]
agent_rows = []
for llm_calls in [1, 3, 5, 10]:
    input_tokens = observed["prompt_tokens"] * llm_calls
    output_tokens = observed["completion_tokens"] * llm_calls
    estimated_cost = (
        input_tokens / 1_000_000 * INPUT_PRICE_PER_MILLION
        + output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MILLION
    )
    agent_rows.append({
        "LLM calls per user request": llm_calls,
        "estimated input tokens": input_tokens,
        "estimated output tokens": output_tokens,
        "estimated total tokens": input_tokens + output_tokens,
        "estimated cost": estimated_cost,
    })

agent_df = pd.DataFrame(agent_rows)
display(agent_df)

### Optional real agent-loop experiment

Set the flag to `True` only when you intentionally want three additional model calls. This is a controlled demonstration, not a production agent.

In [ ]:
RUN_REAL_AGENT_LOOP = False

if RUN_REAL_AGENT_LOOP:
    loop_tasks = [
        "Create a two-step plan for explaining token optimization.",
        "Critique this plan and identify one missing consideration.",
        "Give the final explanation in three concise bullet points.",
    ]
    start_index = len(experiment_rows)
    for step, task in enumerate(loop_tasks, start=1):
        call_chat(
            f"agent_loop_step_{step}",
            [{"role": "user", "content": task}],
            max_tokens=120,
        )
    real_loop_df = pd.DataFrame(experiment_rows[start_index:])
    display(real_loop_df[["experiment", "prompt_tokens", "completion_tokens", "total_tokens"]])
    print("Real loop total tokens:", real_loop_df["total_tokens"].sum())
else:
    print("Skipped. Set RUN_REAL_AGENT_LOOP=True to run it.")

## 9. Final dashboard

Use this table as the closing slide of the live demo. It also exposes request IDs that can be searched in APISIX/Grafana when the corresponding headers are available.

In [ ]:
final_df = add_cost_columns(results_table())
display(final_df[[
    "experiment", "prompt_tokens", "completion_tokens", "total_tokens",
    "ttft_s", "total_latency_s", "tokens_per_second",
    "finish_reason", "request_cost", "demo_request_id", "gateway_request_id"
]])

## 10. Closing message for the audience

**Token optimization is not simply reducing every prompt. It is allocating the smallest useful context and output budget that preserves answer quality.**

Production controls to mention:

- Limit retrieved RAG chunks and rerank them.
- Summarize or window long conversation history.
- Set task-specific output ceilings.
- Cache repeated prompts and shared prefixes where supported.
- Put hard iteration limits and exit conditions on agents.
- Route simple tasks to smaller models.
- Monitor tokens by model, application, user/team, status, and trace/request ID.

### Important interpretation notes

- One run is illustrative, not a benchmark. Repeat runs and report percentiles for performance testing.
- End-to-end latency includes gateway, network, queue, prefill, generation, and client overhead.
- Non-streaming latency does not reveal TTFT.
- More context does not always improve quality; irrelevant context can make answers worse.
- With streaming, capture usage from the final SSE event or instrument the inference service. A gateway body logger alone may not reconstruct the complete stream.